# Advanced RAG in one notebook: the running example, end to end

*Are the directors of Move (1970) and Méditerranée (1963) from the same country?*

This notebook walks the question through every technique in the repo, using `ragkit` (standard library only). Run it from `code/notebooks/`. Each section links to the chapter that explains it.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../examples"))
from corpus import DOCS, QUESTION, GOLD
import ragkit
for i, d in enumerate(DOCS): print(i, d[:90])

## 1. Naive RAG fails ([docs/02](../../docs/02-failure-modes.md))
One query cannot reach the directors' nationalities.

In [ ]:
hr = ragkit.HybridRetriever(DOCS)
for i, s in hr.search(QUESTION, 3): print(f"{s:.3f}", DOCS[i][:80])

## 2. Route first ([docs/14](../../docs/14-long-context-vs-rag.md))

In [ ]:
router = ragkit.HeuristicRouter()
for q in [QUESTION, "Who directed Move (1970)?", "what is 17 * 3?"]:
    print(router.route(q), "<-", q)

## 3. When to retrieve: QuCo-RAG's corpus statistics ([paper note](../../papers/2026/quco-rag.md))

In [ ]:
pretrain = DOCS[:2] * 50 + DOCS[2:4] * 2 + DOCS[6:]
trig = ragkit.QuCoTrigger(ragkit.InMemoryCorpusStats(pretrain), tau_entity=20)
print(trig.before_generation(QUESTION))
print(trig.during_generation([("Jean-Daniel Pollet", "was born in", "Brazil")]))

## 4. The Search-R1 protocol, and what RL sees ([docs/04](../../docs/04-learned-retrieval-policies.md))

In [ ]:
turns = iter([
    "<search>Move 1970 film director</search>",
    "<search>Méditerranée 1963 film director</search>",
    "<search>Stuart Rosenberg nationality</search>",
    "<search>Jean-Daniel Pollet nationality</search>",
    "<answer>no</answer>"])
traj = ragkit.AgenticRAG(lambda p: next(turns), hr.retrieve_text, top_k=1).run(QUESTION)
for s in traj.steps: print("MODEL" if s.is_model_generated else "ENV  ", s.text.strip()[:90])
mask = ragkit.token_mask_from_steps(traj.steps)
print("tokens trained on:", sum(mask), "of", len(mask))

In [ ]:
# GRPO over a group of rollouts with outcome reward
rewards = [1, 0, 1, 0, 1, 0]
print([round(a, 2) for a in ragkit.group_relative_advantages(rewards)])

## 5. How much to search: AutoSearch's reward ([paper note](../../papers/2026/autosearch.md))

In [ ]:
for inter in (["?", "?", "?", "no"], ["?", "?", "?", "no", "no", "no"], ["?", "?"]):
    r = ragkit.autosearch_reward(inter, inter[-1], GOLD)
    print(len(inter), "searches ->", r["regime"], round(r["total"], 3))

## 6. What to retrieve: graphs, trees and triplets ([docs/07](../../docs/07-knowledge-representation.md))

In [ ]:
import importlib.util
def load(name):
    spec = importlib.util.spec_from_file_location(name, f"../examples/{name}.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
ppr_ex = load("05_graph_ppr")
g = ragkit.TripletGraph(ppr_ex.TRIPLETS)
print("PPR:", g.rank_passages(["Move (1970)", "Méditerranée (1963)"], k=4))

In [ ]:
t2 = load("04_t2rag")
ans, state, log = ragkit.T2RAG(t2.decompose, hr.retrieve_text, t2.resolve, t2.answer).run(QUESTION)
for row in log: print(row)
print("answer:", ans)

In [ ]:
tree = ragkit.RaptorTree(DOCS, branching=3)
print("depth", tree.depth)
for i, lvl, text in tree.search("French cinema movement", 3): print(lvl, text[:90])

## 7. Use it well: rerank, diversify, order ([docs/01](../../docs/01-foundations.md))

In [ ]:
q = "Jean-Daniel Pollet nationality French filmmaker"
cands = [i for i, _ in hr.search(q, 8)]
rr = [cands[j] for j, _ in ragkit.ProximityReranker().rerank(q, [DOCS[i] for i in cands])]
print("reranked:", rr, "-> reader order:", ragkit.lost_in_the_middle_order(rr[:4]))

## 8. At what cost: a *safe* answer cache ([docs/10](../../docs/10-efficiency.md))

In [ ]:
cache = ragkit.GroundedAnswerCache(sim_threshold=0.8)
cache.put("Who directed Move (1970)?", "Stuart Rosenberg", ["p0"], {"p0": 1})
print(cache.get("Who directed Move (1970)?", {"p0": DOCS[0]}, {"p0": 1}))
print(cache.get("Who directed Move (1970)?", {"p0": DOCS[0]}, {"p0": 2}))  # document changed

## 9. Evaluate honestly ([docs/11](../../docs/11-evaluation.md))

In [ ]:
import json
recs = [json.loads(l) for l in open("../examples/data/toy_run.jsonl")]
print(ragkit.evaluate(recs, ks=(1, 5)))

## Next
- Swap `lambda p: next(turns)` for `ragkit.OpenAICompatibleLLM(...)` pointed at a vLLM server to run a real model.
- Reproduce the papers with their official code: [experiments/](../../experiments/README.md).